# PDF Question Answering (RAG)
Hugging Face embeddings (local) + Pinecone (vector store) + Groq (answers).

**Before running:** create a file named `.env` next to this notebook:
```
PINECONE_API_KEY=pcsk_...
GROQ_API_KEY=gsk_...
```
Never paste keys into notebook cells.

In [ ]:
# 1. Install (run once, then restart the kernel)
%pip install -q langchain-community langchain-text-splitters langchain-huggingface langchain-pinecone langchain-groq pinecone pypdf sentence-transformers python-dotenv ipywidgets

In [ ]:
# 2. Imports and keys
import os
import time
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore
from langchain_groq import ChatGroq

load_dotenv(find_dotenv(usecwd=True), override=True)

for name in ("PINECONE_API_KEY", "GROQ_API_KEY"):
    if not os.environ.get(name):
        raise RuntimeError(f"{name} is missing. Add it to the .env file next to this notebook and re-run this cell.")
print("Keys loaded.")

In [ ]:
# 3. Settings
PDF_DIR = r"D:\projects\LLMAPP\documents"   # folder that contains budget_speech.pdf
INDEX_NAME = "langchain-demo"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"   # 384 dimensions
LLM_MODEL = "openai/gpt-oss-20b"                        # Groq model

In [ ]:
# 4. Load the PDFs and split them into chunks
def read_doc(directory):
    return PyPDFDirectoryLoader(directory).load()


def chunk_data(docs, chunk_size=800, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    return splitter.split_documents(docs)


pages = read_doc(PDF_DIR)
chunks = chunk_data(pages)
print(len(pages), "pages ->", len(chunks), "chunks")

In [ ]:
# 5. Embedding model (runs locally, no API key)
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
dimension = len(embeddings.embed_query("test"))
print("Embedding dimension:", dimension)   # 384

In [ ]:
# 6. Create the Pinecone index if it does not exist
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=INDEX_NAME,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(1)

existing_dim = pc.describe_index(INDEX_NAME).dimension
assert existing_dim == dimension, f"Index has dimension {existing_dim} but the embeddings have {dimension}. Use a new INDEX_NAME."
print("Index ready:", INDEX_NAME)

In [ ]:
# 7. Upload the CHUNKS (clears old vectors first, so re-running never creates duplicates)
index = pc.Index(INDEX_NAME)
try:
    index.delete(delete_all=True)
except Exception as e:
    print("Nothing to clear:", e)

vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name=INDEX_NAME,
    pinecone_api_key=os.environ["PINECONE_API_KEY"],
)

time.sleep(5)   # stats can lag a few seconds
print(index.describe_index_stats())   # total_vector_count should equal the number of chunks

In [ ]:
# 8. LLM (Groq)
llm = ChatGroq(model=LLM_MODEL, temperature=0, max_tokens=1024)
print(llm.invoke("Say hello").content)

In [ ]:
# 9. Retrieval + answering
def retrieve_query(query, k=5):
    return vectorstore.similarity_search(query, k=k)


def retrieve_answers(query, k=5, show_sources=True):
    docs = retrieve_query(query, k=k)
    context = "\n\n".join(
        f"[PDF page {d.metadata.get('page', 0) + 1}] {d.page_content}" for d in docs
    )
    prompt = f"""You answer questions about a document using only the context below.
Quote numbers exactly as written, including units such as "lakh crore" or "crore".
If the answer is not in the context, say "I don't know".

Context:
{context}

Question: {query}
Answer:"""
    answer = llm.invoke(prompt).content
    if show_sources:
        pages_used = sorted({d.metadata.get("page", 0) + 1 for d in docs})
        print("Sources (PDF pages):", pages_used)
    return answer

In [ ]:
# 10. Ask questions
print(retrieve_answers("How much will the agriculture credit target be increased to?"))

In [ ]:
print(retrieve_answers("What is the new rebate limit in the new tax regime?"))